Imports

In [112]:
import pandas as pd
import numpy as np
import os
import kagglehub
import re
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

Loading and Combining the Data

In [113]:
# Download dataset
path = kagglehub.dataset_download(
    "francescogreco97/human-llm-generated-phishing-legitimate-emails"
)

# Paths
human_legit_path = os.path.join(path, "human-generated", "legit.csv")
human_phishing_path = os.path.join(path, "human-generated", "phishing.csv")
llm_legit_path = os.path.join(path, "llm-generated", "legit.csv")
llm_phishing_path = os.path.join(path, "llm-generated", "phishing.csv")

# Function to clean text
def clean_text(text):
    return str(text).replace("\r", " ").replace("\t", " ").replace("\n"," ").strip()

#Load Human Data (only 'body')
df_human_legit = pd.read_csv(human_legit_path)[['body']]
df_human_phishing = pd.read_csv(human_phishing_path)[['body']]

df_human_legit['body'] = df_human_legit['body'].apply(clean_text)
df_human_phishing['body'] = df_human_phishing['body'].apply(clean_text)

# Combine Human data
df_human = pd.concat([df_human_legit, df_human_phishing], ignore_index=True)
df_human = df_human.rename(columns={'body': 'text'})  # unify column name
df_human['label'] = 1  # Human = 1

#Load LLM Data (only 'text')
df_llm_legit = pd.read_csv(llm_legit_path)[['text']]
df_llm_phishing = pd.read_csv(llm_phishing_path, on_bad_lines='skip')[['text']]
# I have to skip on 405 lines of the llm phising data because of some problem in the data that I unfortunately cant fix.
print("Lines skipped:", 1000-df_llm_phishing.shape[0])

df_llm_legit['text'] = df_llm_legit['text'].apply(clean_text)
df_llm_phishing['text'] = df_llm_phishing['text'].apply(clean_text)

# Combine LLM data
df_llm = pd.concat([df_llm_legit, df_llm_phishing], ignore_index=True)
df_llm['label'] = 0  # LLM

#Combine Human + LLM into one dataset
df_all = pd.concat([df_human, df_llm], ignore_index=True)

X_features = df_all[['text']]
y_labels = df_all['label'].values

Using Colab cache for faster access to the 'human-llm-generated-phishing-legitimate-emails' dataset.
Lines skipped: 405


Feature Extraction

In [114]:
def extract_features(df, text_column):
    features = pd.DataFrame()

    # Text length
    features['char_length'] = df[text_column].apply(len)

    # Word count
    features['word_count'] = df[text_column].apply(lambda x: len(x.split()))

    # URL count
    url_regex = r'https?://\S+|www\.\S+'
    features['url_count'] = df[text_column].apply(lambda x: len(re.findall(url_regex, x)))

    # Has URL or not
    features['has_url'] = (features['url_count'] > 0).astype(int)

    features['sent_len_std'] = df[text_column].apply(lambda x: np.std([len(s.split()) for s in re.split(r'[.!?]+', str(x)) if s.strip()]) if len(re.split(r'[.!?]+', str(x))) > 1 else 0)

    features['all_caps_words'] = df[text_column].apply(lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 1))

    #### Not important features (removed) ####

    # Average word length
    #features['avg_word_length'] = df[text_column].apply(
     #   lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0
    #)

    #features['digit_density'] = df[text_column].apply(lambda x: sum(c.isdigit() for c in str(x)) / len(str(x)) if len(str(x)) > 0 else 0)

    #features['newline_density'] = df[text_column].apply(lambda x: str(x).count('\n') / len(str(x)) if len(str(x)) > 0 else 0)

    #def calculate_ari(text):
    #  words = text.split()
    #  if not words: return 0
    #  characters = len(text)
    #  sentences = len(re.split(r'[.!?]+', text))
    #  return 4.71 * (characters / len(words)) + 0.5 * (len(words) / sentences) - 21.43
    #features['readability_score'] = df[text_column].apply(lambda x: calculate_ari(str(x)))

    return features

X_features_num = extract_features(df_all, 'text')
y_labels = df_all['label'].values  # already 0 = LLM, 1 = Human


Train One-Class SVM

In [115]:
# Split for training/testing (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_features_num, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training
svc = SVC(kernel='linear')
svc.fit(X_train_scaled, y_train)

SVC(kernel='linear')

Predict and evaluate

In [116]:
y_pred = svc.predict(X_test_scaled)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Full report
print(classification_report(y_test, y_pred, target_names=['LLM', 'Human']))

Confusion Matrix:
 [[299  20]
 [ 36 364]]
              precision    recall  f1-score   support

         LLM       0.89      0.94      0.91       319
       Human       0.95      0.91      0.93       400

    accuracy                           0.92       719
   macro avg       0.92      0.92      0.92       719
weighted avg       0.92      0.92      0.92       719



Feature importance

In [117]:
# Extract Feature Importance from the SVM
importance = svc.coef_[0]
feature_names = X_train.columns

# Create a DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print("SVM Feature Weights:")
print(feature_importance_df)

SVM Feature Weights:
          Feature  Importance
5  all_caps_words    6.096645
2       url_count    5.789515
0     char_length    4.804733
4    sent_len_std    3.972493
3         has_url   -1.551673
1      word_count   -3.166938
